# PyTorch DataLoader for DST Data

This notebook teaches you how to create a **DataLoader** for efficiently loading variable-length DST data into PyTorch for machine learning.

## What is a DataLoader?

A **DataLoader** is a tool that:
1. **Loads data in batches** - Instead of loading all data at once, load small groups (e.g., 32 events)
2. **Shuffles data** - Randomizes order for better training
3. **Works with GPU** - Automatically moves data to GPU if available
4. **Handles complex data** - Can work with graphs, variable-length sequences, etc.

### Why Do We Need It?

```python
# ❌ BAD: Load all data at once (runs out of memory!)
all_events = parse_1000_files()  # Too big for RAM!
model.train(all_events)

# ✓ GOOD: Load data in small batches
dataloader = DataLoader(dataset, batch_size=32)
for batch in dataloader:
    model.train(batch)  # Only 32 events in memory at a time
```

## How DataLoader Works

```
HDF5 File on Disk    →    Dataset    →    DataLoader    →    Training Loop
(100 GB, 10000 events)    (Indexer)      (Batching)         (32 events/batch)
```

### Key Concepts:

1. **Dataset**: Knows how to get ONE event by index
   ```python
   dataset[0]  # Returns event 0
   dataset[42] # Returns event 42
   ```

2. **DataLoader**: Automatically batches multiple events
   ```python
   loader = DataLoader(dataset, batch_size=32)
   for batch in loader:
       # batch contains 32 events combined
   ```

3. **Offset System**: Variable-length data needs special handling
   - Each event has different number of hits
   - Use `hit_offsets` to slice the right data

## Setup

**Task**: Import necessary libraries

**Hint**: You'll need:
- `torch` for PyTorch tensors
- `h5py` for reading HDF5 files
- `numpy` for arrays
- `Dataset` from `torch.utils.data` - base class for datasets
- `DataLoader` from `torch_geometric.loader` - for batching graph data

In [ ]:
# Your code here
import torch
import h5py
import numpy as np
# Add more imports

## Part 1: Understanding the HDF5 Structure

### Exercise 1.1: Inspect the HDF5 File

**Task**: Open the variable-length HDF5 file you created in previous exercises and print its structure

**Hints**:
1. Use `h5py.File(path, 'r')` to open
2. Print `file.keys()` to see available datasets
3. For each key, print the shape: `file[key].shape`
4. Close the file when done

In [ ]:
# Path to your HDF5 file from previous exercise
hdf5_path = "/ceph/work/SATORI/antonpr/ml/benMC/sdanalysis_2019/handson/handson_new/exercise_vlen_output.h5"

# Your code here

### Exercise 1.2: Understanding Offsets

**Task**: Calculate statistics about the offset structure

**Hints**:
1. Load `hit_offsets` array
2. Number of events = `len(hit_offsets) - 1`
3. Total hits = `hit_offsets[-1]`
4. Hits per event = `np.diff(hit_offsets)`
5. Print min, max, mean hits per event

In [ ]:
# Your code here

## Part 2: Creating a Simple Dataset Class

### What is a Dataset Class?

A Dataset class has THREE required methods:

```python
class MyDataset(Dataset):
    def __init__(self, path):
        # Initialize: open file, store path, etc.
        pass
    
    def __len__(self):
        # Return total number of events
        return num_events
    
    def __getitem__(self, idx):
        # Return data for event at index idx
        return event_data
```

### Exercise 2.1: Implement `__init__` Method

**Task**: Create the initialization method that opens the HDF5 file and stores references to datasets

**Hints**:
1. Store the file path
2. Open HDF5 file: `h5py.File(path, 'r')`
3. Store references to datasets: `self.pulse_area = f['pulse_area']`
4. Store: `pulse_area`, `detector_positions`, `hit_offsets`, `time_traces`, `hit_tt_offsets`, `nfold`
5. Optionally store labels: `energy`, `shower_axis`, `shower_core`

In [ ]:
from torch.utils.data import Dataset

class VlenDSTDataset(Dataset):
    def __init__(self, path):
        """
        Initialize dataset from HDF5 file
        
        Args:
            path: Path to HDF5 file created by parse_dst_file_vlen
        """
        # Your code here
        self.path = path
        
        # Open file and store references
        self.file = h5py.File(path, 'r')
        
        # Store dataset references (NOT loaded to memory yet!)
        # self.pulse_area = ...
        # self.detector_positions = ...
        # ...
        
    def __len__(self):
        # We'll implement this in the next exercise
        pass
    
    def __getitem__(self, idx):
        # We'll implement this in the next exercise
        pass
    
    def __del__(self):
        # Clean up: close file when dataset is deleted
        if hasattr(self, 'file') and self.file is not None:
            self.file.close()

# Test your implementation
# dataset = VlenDSTDataset(hdf5_path)
# print(f"File opened successfully!")
# print(f"Available datasets: {list(dataset.file.keys())}")

### Exercise 2.2: Implement `__len__` Method

**Task**: Return the total number of events in the dataset

**Hint**: Number of events = `len(self.hit_offsets) - 1`

In [ ]:
# Add this method to your VlenDSTDataset class above:

def __len__(self):
    """
    Return total number of events
    """
    # Your code here
    pass

# Test
# dataset = VlenDSTDataset(hdf5_path)
# print(f"Total events: {len(dataset)}")

### Exercise 2.3: Implement `__getitem__` Method - Hit-Level Data

**Task**: Extract hit-level data for a single event

**Hints**:
1. Get hit boundaries:
   ```python
   start_hit = self.hit_offsets[idx]
   end_hit = self.hit_offsets[idx + 1]
   ```
2. Slice hit-level arrays:
   ```python
   pulse_area = self.pulse_area[start_hit:end_hit]
   positions = self.detector_positions[start_hit:end_hit]
   ```
3. Convert to PyTorch tensors:
   ```python
   pulse_area = torch.from_numpy(pulse_area).float()
   ```
4. Return a dictionary with all data

In [ ]:
# Add this method to your VlenDSTDataset class:

def __getitem__(self, idx):
    """
    Get data for event at index idx
    
    Args:
        idx: Event index (0 to len-1)
        
    Returns:
        dict with event data
    """
    # Your code here
    
    # 1. Get hit boundaries
    # start_hit = ...
    # end_hit = ...
    
    # 2. Extract hit-level data
    # pulse_area = ...
    # positions = ...
    # detector_ids = ...
    # nfold = ...
    
    # 3. Convert to tensors
    # pulse_area = torch.from_numpy(pulse_area[...]).float()
    
    # 4. Return dictionary
    return {
        'pulse_area': pulse_area,
        'detector_positions': positions,
        'detector_ids': detector_ids,
        'nfold': nfold,
    }

# Test
# dataset = VlenDSTDataset(hdf5_path)
# event0 = dataset[0]
# print(f"Event 0 data:")
# for key, value in event0.items():
#     print(f"  {key}: {value.shape}")

### Exercise 2.4: Add Time Trace Data

**Task**: Extend `__getitem__` to also load time traces

**Hints**:
1. Get time trace boundaries using `hit_tt_offsets`:
   ```python
   tt_start = self.hit_tt_offsets[start_hit]
   tt_end = self.hit_tt_offsets[end_hit]
   ```
2. Slice time traces: `self.time_traces[tt_start:tt_end]`
3. Convert to tensor and add to return dictionary
4. Verify: sum of nfold should equal number of time trace windows

In [ ]:
# Modify your __getitem__ method to include time traces
# Your code here

### Exercise 2.5: Add Labels (Optional)

**Task**: Add event-level labels (energy, shower_axis, shower_core) to the output

**Hints**:
1. These are event-level arrays, so just index with `idx`:
   ```python
   energy = self.energy[idx]
   ```
2. Check if label exists before accessing:
   ```python
   if 'energy' in self.file:
       energy = torch.tensor(self.energy[idx]).float()
   ```

In [ ]:
# Modify your __getitem__ method to include labels
# Your code here

## Part 3: Testing Your Dataset

### Exercise 3.1: Create Dataset and Test Basic Access

**Task**: Create a dataset instance and access a few events

**Hints**:
1. Create dataset: `dataset = VlenDSTDataset(hdf5_path)`
2. Print length: `len(dataset)`
3. Access first event: `dataset[0]`
4. Access random event: `dataset[42]`
5. Print shapes of all fields

In [ ]:
# Your code here

### Exercise 3.2: Verify Data Consistency

**Task**: For several events, verify that the data is consistent

**Hints**:
1. Check: number of pulse_area rows = number of positions
2. Check: sum of nfold = number of time trace windows
3. Check: time_traces shape is (n_windows, 2, 128)
4. Loop over first 10 events and verify

In [ ]:
# Your code here

## Part 4: Creating a DataLoader

### What Happens in Batching?

When you batch variable-length data:
```python
Event 0: 5 hits   [x x x x x]
Event 1: 3 hits   [x x x]
Event 2: 7 hits   [x x x x x x x]

Batch: 15 hits total [x x x x x | x x x | x x x x x x x]
       with offsets  [0         5       8               15]
```

PyTorch Geometric's DataLoader does this automatically!

### Exercise 4.1: Create a Simple DataLoader

**Task**: Create a DataLoader with batch size 4

**Hints**:
1. Import: `from torch_geometric.loader import DataLoader`
2. Import: `from torch_geometric.data import Data`
3. You'll need to modify `__getitem__` to return a `Data` object instead of dict
4. Create loader:
   ```python
   loader = DataLoader(dataset, batch_size=4, shuffle=True)
   ```

In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# Modify __getitem__ to return Data object
# Instead of returning dict, return:
# return Data(
#     x=node_features,  # Shape: (num_hits, num_features)
#     time_traces=time_traces,
#     nfold=nfold,
#     y=labels  # Event-level label
# )

# Your code here

### Exercise 4.2: Prepare Node Features

**Task**: Combine hit-level data into node features for the graph

**Hints**:
1. Node features should include:
   - Detector positions (3 values: x, y, z)
   - Pulse area for lower and upper PMT (2 values)
   - Total: 5 features per node
2. Concatenate: `torch.cat([positions, pulse_area], dim=1)`
3. Shape should be: `(num_hits, 5)`

In [ ]:
# Modify __getitem__ to create node features
# Your code here

### Exercise 4.3: Create and Test DataLoader

**Task**: Create a DataLoader and iterate through one batch

**Hints**:
1. Create loader with `batch_size=4`
2. Iterate: `for batch in loader:`
3. Print batch attributes:
   - `batch.x` - node features (all hits from all events in batch)
   - `batch.batch` - which event each hit belongs to
   - `batch.y` - labels for each event in batch
4. Break after first batch

In [ ]:
# Your code here

### Exercise 4.4: Understanding the Batch

**Task**: Analyze the batch structure

**Hints**:
1. Print `batch.x.shape` - total hits from all events
2. Print `batch.batch` - event index for each hit
3. Print `batch.y` - one label per event
4. Verify: `len(batch.y)` should equal `batch_size`
5. Count hits per event: `torch.bincount(batch.batch)`

In [ ]:
# Your code here

## Part 5: Complete Dataset Implementation

### Exercise 5.1: Put It All Together

**Task**: Write the complete dataset class with all features

**Your class should include**:
1. `__init__`: Open file, store dataset references
2. `__len__`: Return number of events
3. `__getitem__`: 
   - Extract hit-level data using offsets
   - Extract time traces using two-level offsets
   - Combine into node features
   - Return PyG `Data` object
4. `__del__`: Close file properly

In [ ]:
class VlenDSTDataset(Dataset):
    """
    PyTorch Dataset for variable-length DST data stored in HDF5
    
    Usage:
        dataset = VlenDSTDataset('data.h5')
        loader = DataLoader(dataset, batch_size=32, shuffle=True)
        
        for batch in loader:
            # batch.x has node features
            # batch.batch indicates which event each node belongs to
            # batch.y has labels
    """
    
    def __init__(self, path):
        # Your code here
        pass
    
    def __len__(self):
        # Your code here
        pass
    
    def __getitem__(self, idx):
        # Your code here
        pass
    
    def __del__(self):
        # Your code here
        pass

### Exercise 5.2: Test Full Pipeline

**Task**: Test your complete dataset with a DataLoader

**Hints**:
1. Create dataset
2. Create loader with batch_size=8, shuffle=True
3. Iterate through a few batches
4. Print statistics for each batch
5. Verify data makes sense